# True DLinear Re-run v2

Replaces the plain-linear "DLinear" baseline with the correct Zeng et al. (2023)
decomposition model: a moving-average trend branch and a remainder branch, each a
separate `Linear(seq_len, pred_len)`, summed at output. Weights are shared across
variates (channel-independent), matching the paper description.

**Produces two CSVs with schemas identical to their originals:**
- `results_grid_dlinear_v2.csv` — 27 runs, schema matches `results_grid_canonical.csv`
- `results_lf_dlinear_v2.csv`  — 12 runs, schema matches `results_leader_follower_v3.csv`

**Scope:** DLinear only. CI and CD are not touched.

**Runtime:** ~30 min on a single T4 (39 runs, each ~45 s). No memory pressure.

**Checkpointing:** atomic CSV write after every run; safe to interrupt and resume.


In [ ]:
import gc
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# DLinear is fully CPU-feasible, but runs faster on GPU if present.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    """Seed all RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
class TrueDLinear(nn.Module):
    """DLinear as in Zeng et al. (2023), Are Transformers Effective for Time Series?

    Decomposes each variate's lookback window into a moving-average trend and a
    remainder, then maps each component independently to the forecast horizon via
    a separate Linear layer. The two outputs are summed. Weights are shared across
    variates (channel-independent). No warmup or Transformer components.

    Input:  (B, seq_len, C)
    Output: (B, pred_len, C)
    """

    def __init__(self, seq_len: int, pred_len: int, ma_kernel: int = 25) -> None:
        super().__init__()
        if ma_kernel % 2 == 0:
            raise ValueError(f"ma_kernel must be odd for symmetric padding; got {ma_kernel}")
        padding = (ma_kernel - 1) // 2
        # AvgPool1d treats the length dimension as the sequence axis.
        # Input to pool: (B*C, 1, seq_len) -> output: (B*C, 1, seq_len)
        self.avg_pool = nn.AvgPool1d(kernel_size=ma_kernel, stride=1, padding=padding)
        # Two independent branches: trend and remainder.
        self.linear_trend     = nn.Linear(seq_len, pred_len)
        self.linear_remainder = nn.Linear(seq_len, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, seq_len, C) -> (B, pred_len, C)"""
        B, L, C = x.shape
        # Reshape to (B*C, 1, L) so AvgPool1d operates along the time axis.
        x_flat = x.permute(0, 2, 1).reshape(B * C, 1, L)   # (B*C, 1, L)
        trend  = self.avg_pool(x_flat).reshape(B * C, L)    # (B*C, L)
        # Clamp to input length in case padding produces off-by-one.
        trend  = trend[:, :L]
        x_flat = x_flat.reshape(B * C, L)
        remainder = x_flat - trend                           # (B*C, L)
        out = (self.linear_trend(trend)
               + self.linear_remainder(remainder))           # (B*C, pred_len)
        return out.reshape(B, C, -1).permute(0, 2, 1)       # (B, pred_len, C)


# ── Architecture sanity checks ────────────────────────────────────────────────
_m = TrueDLinear(seq_len=512, pred_len=96, ma_kernel=25)
# Output shape
_x = torch.zeros(4, 512, 21)
_y = _m(_x)
assert _y.shape == (4, 96, 21), f"Wrong output shape: {_y.shape}"
# Two independent branches (different parameter objects)
assert _m.linear_trend is not _m.linear_remainder, "Branches must be independent"
# Padding must not change sequence length
_xp = torch.zeros(2, 1, 512)
_tp = _m.avg_pool(_xp)
assert _tp.shape[-1] == 512, f"AvgPool1d changed length: {_tp.shape}"
del _m, _x, _y, _xp, _tp
print("TrueDLinear architecture checks passed.")
print("  Trend branch:     nn.Linear(512, 96)  -- independent weights")
print("  Remainder branch: nn.Linear(512, 96)  -- independent weights")
print("  MA kernel: 25, padding: 12 (symmetric, output length == input length)")


In [ ]:
# ── Shared constants (match original runs exactly) ────────────────────────────
T_TOTAL:    int   = 14_400
BURN_IN:    int   = 1_000
TRAIN_FRAC: float = 0.6
VAL_FRAC:   float = 0.2
SEQ_LEN:    int   = 512
PRED_LEN:   int   = 96
MA_KERNEL:  int   = 25
LR:         float = 1e-4
WEIGHT_DECAY: float = 1e-4
MAX_EPOCHS: int   = 50
PATIENCE:   int   = 10
SEEDS: list[int]  = [42, 123, 456]


def split_and_normalise(
    data: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """60/20/20 split with per-channel z-score fit on train only."""
    T       = len(data)
    n_train = int(T * TRAIN_FRAC)
    n_val   = int(T * VAL_FRAC)
    train   = data[:n_train]
    val     = data[n_train: n_train + n_val]
    test    = data[n_train + n_val:]
    mean    = train.mean(axis=0, keepdims=True)
    std     = train.std(axis=0, keepdims=True)
    std     = np.where(std == 0, 1.0, std)
    return (train - mean) / std, (val - mean) / std, (test - mean) / std


def make_windows(
    data: np.ndarray, seq_len: int, pred_len: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Sliding-window (x, y) pairs.  Uses stride tricks: no Python loop."""
    T = len(data)
    n = T - seq_len - pred_len + 1
    if n <= 0:
        raise ValueError(
            f"Not enough timesteps ({T}) for seq_len+pred_len={seq_len + pred_len}."
        )
    # Shape: (n, seq_len+pred_len, C) via a single strided view.
    C     = data.shape[1]
    itemsize = data.strides[0]
    view  = np.lib.stride_tricks.as_strided(
        data,
        shape=(n, seq_len + pred_len, C),
        strides=(itemsize, itemsize, data.strides[1]),
    )
    xs = np.ascontiguousarray(view[:, :seq_len])
    ys = np.ascontiguousarray(view[:, seq_len:])
    return (torch.tensor(xs, dtype=torch.float32),
            torch.tensor(ys, dtype=torch.float32))


def empirical_rho(train_data: np.ndarray) -> float:
    """Mean of upper-triangle pairwise Pearson correlations on training split."""
    corr = np.corrcoef(train_data.T)
    n    = corr.shape[0]
    if n < 2:
        return float("nan")
    i, j = np.triu_indices(n, k=1)
    return float(np.mean(corr[i, j]))


def save_atomic(rows: list[dict], path: Path) -> None:
    """Write CSV atomically so a partial write cannot corrupt existing data."""
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    """Return element-wise mean MSE and MAE over a DataLoader."""
    model.eval()
    mse_sum = mae_sum = 0.0
    n = 0
    for x_batch, y_batch in loader:
        pred     = model(x_batch.to(DEVICE)).cpu()
        mse_sum += nn.functional.mse_loss(pred, y_batch, reduction="sum").item()
        mae_sum += nn.functional.l1_loss(pred, y_batch, reduction="sum").item()
        n       += y_batch.numel()
    return mse_sum / n, mae_sum / n


def train_dlinear(
    datasets: tuple,
    batch_size: int,
    extra_row_fields: dict,
) -> dict:
    """Fit one DLinear configuration; return a result-row dict.

    Args:
        datasets:         (x_tr, y_tr, x_va, y_va, x_te, y_te) tensors.
        batch_size:       Mini-batch size.
        extra_row_fields: Fields to merge into the result row (dataset name,
                          C, rho / gamma, mode, seed, empirical_rho, etc.).
    """
    x_tr, y_tr, x_va, y_va, x_te, y_te = datasets
    train_loader = DataLoader(
        TensorDataset(x_tr, y_tr), batch_size=batch_size,
        shuffle=True, drop_last=False,
    )
    val_loader = DataLoader(
        TensorDataset(x_va, y_va), batch_size=batch_size,
        shuffle=False, drop_last=False,
    )
    test_loader = DataLoader(
        TensorDataset(x_te, y_te), batch_size=batch_size,
        shuffle=False, drop_last=False,
    )
    model     = TrueDLinear(SEQ_LEN, PRED_LEN, MA_KERNEL).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    criterion        = nn.MSELoss()
    steps_per_epoch  = len(train_loader)
    best_val         = float("inf")
    best_epoch       = 0
    best_total_steps = 0
    best_state       = None
    no_improve       = 0
    total_steps      = 0

    try:
        for epoch in range(MAX_EPOCHS):
            model.train()
            for x_batch, y_batch in train_loader:
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x_batch.to(DEVICE)), y_batch.to(DEVICE))
                loss.backward()
                model_params = [p for p in model.parameters() if p.grad is not None]
                nn.utils.clip_grad_norm_(model_params, max_norm=1.0)
                optimizer.step()
                total_steps += 1

            val_mse, _ = evaluate(model, val_loader)
            if val_mse < best_val:
                best_val         = val_mse
                best_epoch       = epoch + 1
                best_total_steps = total_steps
                best_state       = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
    finally:
        # Always load best state if available, even on exception.
        if best_state is not None:
            model.load_state_dict(best_state)
        test_mse, test_mae = evaluate(model, test_loader)
        del model, optimizer
        free_cuda()

    return {
        **extra_row_fields,
        "test_mse":        test_mse,
        "test_mae":        test_mae,
        "best_epoch":      best_epoch,
        "batch_size":      batch_size,
        "steps_per_epoch": steps_per_epoch,
        "total_steps":     best_total_steps,
    }


In [ ]:
# ── Section A: AR(1) synthetic grid ─────────────────────────────────────────
#
# 27 runs: C in {7, 21, 84} x rho in {0.1, 0.5, 0.9} x seeds {42, 123, 456}
# Batch sizes match results_grid_canonical.csv exactly.
# Output schema matches results_grid_canonical.csv.

PHI_GRID:  float = 0.8
C_VALUES:  list[int]   = [7, 21, 84]
RHO_VALUES: list[float] = [0.1, 0.5, 0.9]

# Batch sizes from the original grid (Table 3).
BATCH_BY_C: dict[int, int] = {7: 128, 21: 128, 84: 32}

GRID_OUT = Path("/kaggle/working/results_grid_dlinear_v2.csv")
GRID_TMP = GRID_OUT.with_suffix(".csv.tmp")


def generate_ar1(C: int, phi: float, rho: float, seed: int) -> np.ndarray:
    """AR(1) with compound-symmetry covariance; matches the original grid generator."""
    rng   = np.random.default_rng(seed)
    Sigma = np.full((C, C), rho, dtype=np.float64)
    np.fill_diagonal(Sigma, 1.0)
    L     = np.linalg.cholesky(Sigma)
    total = T_TOTAL + BURN_IN
    X     = np.zeros((total, C), dtype=np.float64)
    for t in range(1, total):
        X[t] = phi * X[t - 1] + L @ rng.standard_normal(C)
    return X[BURN_IN:]


# ── Resume ────────────────────────────────────────────────────────────────────
def _grid_key(C: int, rho: float, seed: int) -> tuple:
    return (int(C), round(float(rho), 4), int(seed))


if GRID_OUT.exists() and GRID_OUT.stat().st_size > 100:
    _existing  = pd.read_csv(GRID_OUT)
    grid_done  = {_grid_key(r.C, r.rho, r.seed) for r in _existing.itertuples()}
    grid_rows  = _existing.to_dict("records")
    print(f"Grid: resuming, {len(grid_done)} runs already complete.")
else:
    grid_done, grid_rows = set(), []

total_grid = len(C_VALUES) * len(RHO_VALUES) * len(SEEDS)
run_idx    = 0
failures   = []

for C in C_VALUES:
    for rho in RHO_VALUES:
        for seed in SEEDS:
            run_idx += 1
            key = _grid_key(C, rho, seed)
            if key in grid_done:
                print(f"[{run_idx}/{total_grid}] SKIP C={C} rho={rho} seed={seed}")
                continue

            print(f"[{run_idx}/{total_grid}] C={C} rho={rho} seed={seed} ...",
                  end=" ", flush=True)
            t0 = time.time()
            try:
                set_seed(seed)
                raw                          = generate_ar1(C, PHI_GRID, rho, seed)
                train_d, val_d, test_d       = split_and_normalise(raw)
                emp_rho                      = empirical_rho(train_d)
                datasets                     = (
                    *make_windows(train_d, SEQ_LEN, PRED_LEN),
                    *make_windows(val_d,   SEQ_LEN, PRED_LEN),
                    *make_windows(test_d,  SEQ_LEN, PRED_LEN),
                )
                extra = {
                    "dataset":      "synthetic_ar1",
                    "C":            C,
                    "rho":          rho,
                    "empirical_rho": emp_rho,
                    "mode":         "DLinear",
                    "pred_len":     PRED_LEN,
                    "seed":         seed,
                }
                row = train_dlinear(datasets, BATCH_BY_C[C], extra)
            except Exception as exc:
                free_cuda()
                print(f"FAILED: {type(exc).__name__}: {exc}")
                failures.append((C, rho, seed, repr(exc)))
                continue

            elapsed = time.time() - t0
            print(f"test_mse={row['test_mse']:.4f}  best_epoch={row['best_epoch']}"
                  f"  bs={row['batch_size']}  ({elapsed:.0f}s)")
            grid_rows.append(row)
            grid_done.add(key)
            save_atomic(grid_rows, GRID_OUT)

print(f"\nSection A done. {len(grid_rows)}/{total_grid} runs saved to {GRID_OUT}")
if failures:
    print(f"{len(failures)} failures (rerun cell to retry):")
    for f in failures:
        print(" ", f)


In [ ]:
# ── Section A sanity: compare new DLinear against original plain-linear ───────
import sys
from pathlib import Path

grid_df = pd.read_csv(GRID_OUT)

# Load original for comparison (mount as Kaggle dataset or adjust path)
orig_candidates = [
    "/kaggle/input/results_grid_canonical/results_grid_canonical.csv",
    "/kaggle/working/results_grid_canonical.csv",
]
orig_path = next((p for p in orig_candidates if Path(p).exists()), None)

print("=== True DLinear (v2) mean test MSE by (C, rho) ===")
new_means = (grid_df.groupby(["C", "rho"])["test_mse"].mean().round(4))
print(new_means.to_string())

if orig_path:
    orig = pd.read_csv(orig_path)
    old_dl = orig[orig["mode"] == "DLinear"]
    old_means = old_dl.groupby(["C", "rho"])["test_mse"].mean().round(4)
    print("\n=== Original plain-linear DLinear mean test MSE by (C, rho) ===")
    print(old_means.to_string())
    diff = (new_means - old_means).round(4)
    print("\n=== Diff (true - plain): expect small, typically < 1% ===")
    print(diff.to_string())
else:
    print("\n(original CSV not found; skipping comparison)")


In [ ]:
# ── Section B: leader-follower VAR(1) gamma sweep ────────────────────────────
#
# 12 runs: gamma in {0.0, 0.3, 0.6, 0.9} x seeds {42, 123, 456}
# Batch 128 (matches results_leader_follower_v3.csv DLinear runs).
# Output schema matches results_leader_follower_v3.csv.

PHI_LF:  float = 0.8
RHO_LF:  float = 0.5
C_LF:    int   = 21
GAMMAS:  list[float] = [0.0, 0.3, 0.6, 0.9]
BATCH_LF: int  = 128

N_LEADERS:   int = 10
N_FOLLOWERS: int = 10


def generate_lf(phi: float, gamma: float, rho: float, seed: int) -> np.ndarray:
    """Leader-follower VAR(1), identical generator to train_leader_follower_v3."""
    C = N_LEADERS + N_FOLLOWERS + 1  # 21
    rng   = np.random.default_rng(seed)
    A     = np.zeros((C, C), dtype=np.float64)
    np.fill_diagonal(A, phi)
    for k in range(N_FOLLOWERS):
        A[N_LEADERS + k, k] = gamma
    Sigma = np.full((C, C), rho, dtype=np.float64)
    np.fill_diagonal(Sigma, 1.0)
    L_chol = np.linalg.cholesky(Sigma)
    total  = T_TOTAL + BURN_IN
    X      = np.zeros((total, C), dtype=np.float64)
    for t in range(1, total):
        X[t] = A @ X[t - 1] + L_chol @ rng.standard_normal(C)
    return X[BURN_IN:]


LF_OUT = Path("/kaggle/working/results_lf_dlinear_v2.csv")


def _lf_key(gamma: float, seed: int) -> tuple:
    return (round(float(gamma), 4), int(seed))


if LF_OUT.exists() and LF_OUT.stat().st_size > 100:
    _ex_lf  = pd.read_csv(LF_OUT)
    lf_done = {_lf_key(r.gamma, r.seed) for r in _ex_lf.itertuples()}
    lf_rows = _ex_lf.to_dict("records")
    print(f"LF: resuming, {len(lf_done)} runs already complete.")
else:
    lf_done, lf_rows = set(), []

total_lf = len(GAMMAS) * len(SEEDS)
run_idx  = 0
lf_failures = []

for gamma in GAMMAS:
    for seed in SEEDS:
        run_idx += 1
        key = _lf_key(gamma, seed)
        if key in lf_done:
            print(f"[{run_idx}/{total_lf}] SKIP gamma={gamma} seed={seed}")
            continue

        print(f"[{run_idx}/{total_lf}] gamma={gamma} seed={seed} ...",
              end=" ", flush=True)
        t0 = time.time()
        try:
            set_seed(seed)
            raw                    = generate_lf(PHI_LF, gamma, RHO_LF, seed)
            train_d, val_d, test_d = split_and_normalise(raw)
            datasets               = (
                *make_windows(train_d, SEQ_LEN, PRED_LEN),
                *make_windows(val_d,   SEQ_LEN, PRED_LEN),
                *make_windows(test_d,  SEQ_LEN, PRED_LEN),
            )
            extra = {
                "dataset": "leader_follower_var1",
                "C":       C_LF,
                "rho":     RHO_LF,
                "gamma":   gamma,
                "mode":    "DLinear",
                "seed":    seed,
            }
            row = train_dlinear(datasets, BATCH_LF, extra)
        except Exception as exc:
            free_cuda()
            print(f"FAILED: {type(exc).__name__}: {exc}")
            lf_failures.append((gamma, seed, repr(exc)))
            continue

        elapsed = time.time() - t0
        print(f"test_mse={row['test_mse']:.4f}  best_epoch={row['best_epoch']}"
              f"  bs={row['batch_size']}  ({elapsed:.0f}s)")
        lf_rows.append(row)
        lf_done.add(key)
        save_atomic(lf_rows, LF_OUT)

print(f"\nSection B done. {len(lf_rows)}/{total_lf} runs saved to {LF_OUT}")
if lf_failures:
    print(f"{len(lf_failures)} failures (rerun cell to retry):")
    for f in lf_failures:
        print(" ", f)


In [ ]:
# ── Section B sanity: compare new DLinear against v3 plain-linear ─────────────
lf_df = pd.read_csv(LF_OUT)

print("=== True DLinear (v2) mean test MSE and CD/CI ratio by gamma ===")
lf_new_means = lf_df.groupby("gamma")["test_mse"].mean().round(4)
print(lf_new_means.to_string())

v3_candidates = [
    "/kaggle/input/results_leader_follower_v3/results_leader_follower_v3.csv",
    "/kaggle/working/results_leader_follower_v3.csv",
]
v3_path = next((p for p in v3_candidates if Path(p).exists()), None)

if v3_path:
    v3 = pd.read_csv(v3_path)
    old_dl_lf   = v3[v3["mode"] == "DLinear"]
    old_lf_means = old_dl_lf.groupby("gamma")["test_mse"].mean().round(4)
    print("\n=== Original plain-linear DLinear means by gamma ===")
    print(old_lf_means.to_string())
    print("\n=== Diff (true - plain) ===")
    print((lf_new_means - old_lf_means).round(4).to_string())
else:
    print("\n(v3 CSV not found; skipping comparison)")


In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(GRID_OUT)))
display(FileLink(str(LF_OUT)))
